main flow

client -> gateway 
            -> schema validation -> httpx client request -> ingestion service -> returns data with status

pydantic schema

In [1]:
from pydantic import BaseModel, Field

class IngestionSchema(BaseModel):
    pdf_file: str = Field(default_factory=str)

connection with ingestion service

In [3]:
import httpx

async def connecting_2_ingestion(payload: IngestionSchema):
    INGESTION_URL = "http://127.0.0.1:8001"

    async with httpx.AsyncClient() as client:
        response = await client.post(url=INGESTION_URL, json=payload.model_dump())

    return response.json(), response.status_code

defining the gateway router

In [4]:
from fastapi import FastAPI, HTTPException, status

app = FastAPI()
pdf_file_path = "/home/rik07/Documents/repos/simple_rag/knowledge_base/raw/attention.pdf"

@app.post(path="/gateway/ingest")
async def handling_ingestion():
    try:
        response_data, status_code = await connecting_2_ingestion(payload=IngestionSchema(pdf_file=pdf_file_path))
        return {
            "data": response_data,
            "status_code": status_code
        }
    except Exception as exp:
        raise HTTPException (
            detail=exp,
            status_code=status.HTTP_404_NOT_FOUND
        )

starting point

In [ ]:
import nest_asyncio, uvicorn, asyncio

nest_asyncio.apply()

if __name__ == "__main__":
    # setting the uvicorn configuration
    config = uvicorn.Config(
        app=app,
        host="127.0.0.1",
        port=8000,
        loop="asyncio"
    )
    # instead of using uvicorn.run(), .Server is used
    server = uvicorn.Server(config=config)

    loop = asyncio.get_event_loop() # to make sure uvicorn use the same server (localhost) as jupyter notebook
    loop.create_task(server.serve())


INFO:     Started server process [51377]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:60354 - "POST /gateway/ingest HTTP/1.1" 200 OK
INFO:     127.0.0.1:42084 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:42084 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:42084 - "POST /gateway/ingest HTTP/1.1" 200 OK
